# Preprocessing Pipeline

Prepares the frozen dataset from `gold_dataset.csv`: integrity checks, light multilingual text cleaning, and three text variants (`text_subject`, `text_body`, `text_subject_body`), saved to `frozen_dataset.csv`.

Labels: topic (`administrative`, `course-exam`, `event`, `deadline-action`, `advertisement`) and priority (`High`, `Medium`, `Low`). The body source column is `body_plain`.

## Step 1 — Imports and configuration

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

INPUT_PATH = Path("../data/processed/gold_dataset.csv")
OUTPUT_PATH = Path("../data/processed/frozen_dataset.csv")

VALID_TOPICS = {
    "administrative",
    "course-exam",
    "event",
    "deadline-action",
    "advertisement",
}

VALID_PRIORITIES = {"High", "Medium", "Low"}

pd.set_option("display.max_colwidth", 120)
print("Input :", INPUT_PATH.resolve())
print("Output:", OUTPUT_PATH.resolve())

Input : D:\UniGe\3\NLP\NLP-project\Preprocessing\gold_dataset.csv
Output: D:\UniGe\3\NLP\NLP-project\Preprocessing\frozen_dataset.csv


## Step 2 — Load the gold dataset

In [ ]:
df_raw = pd.read_csv(INPUT_PATH)

print(f"Loaded rows   : {len(df_raw)}")
print(f"Columns       : {list(df_raw.columns)}")
df_raw.head(3)

Loaded rows   : 919
Columns       : ['id', 'subject', 'body_plain', 'topic', 'priority']


,id,subject,body_plain,topic,priority
0,4e5d730277a0ddab,Benvenuto nel servizio di posta dell'Universita' di Genova,"[Questo e' un messaggio automatico, si prega di non rispondere]\r\n \r\nBenvenuto nel servizio di pos...",administrative,Medium
1,d8a3699e03d7335a,Badge – Student card [TS#142583742],"Gentile studente,\r\n\r\nil tuo badge è disponibile. Lo potrai ritirare presso il Settore Welcome\r\nOffice in Piazz...",administrative,Medium
2,2867eb3c4849e4fb,"AulaWeb2024 90498: ML exam on Thursday Jan, 16th","90498 -> Forum -> Announcements -> ML exam on Thursday Jan, 16th\r\nhttps://2024.aulaweb.unige.it/mod/forum/discuss....",course-exam,High


## Step 3 — Integrity checks

Audit only (nothing dropped yet): row count, missing/invalid topic and priority labels, duplicate `id`s, and empty subjects/bodies.

In [ ]:
def is_blank(series: pd.Series) -> pd.Series:
    """True where the value is NaN or, after str-strip, an empty string."""
    return series.isna() | (series.fillna("").astype(str).str.strip() == "")


report = {
    "rows": len(df_raw),
    "missing_topic": int(df_raw["topic"].isna().sum()),
    "missing_priority": int(df_raw["priority"].isna().sum()),
    "invalid_topic": int((~df_raw["topic"].isin(VALID_TOPICS) & df_raw["topic"].notna()).sum()),
    "invalid_priority": int((~df_raw["priority"].isin(VALID_PRIORITIES) & df_raw["priority"].notna()).sum()),
    "duplicate_ids": int(df_raw["id"].duplicated().sum()),
    "empty_subjects": int(is_blank(df_raw["subject"]).sum()),
    "empty_bodies": int(is_blank(df_raw["body_plain"]).sum()),
    "both_subject_and_body_empty": int(
        (is_blank(df_raw["subject"]) & is_blank(df_raw["body_plain"])).sum()
    ),
}

print("Integrity report")
print("-" * 40)
for k, v in report.items():
    print(f"{k:>30}: {v}")

print("\nTopic distribution:")
print(df_raw["topic"].value_counts(dropna=False).to_string())
print("\nPriority distribution:")
print(df_raw["priority"].value_counts(dropna=False).to_string())

Integrity report
----------------------------------------
                          rows: 919
                 missing_topic: 0
              missing_priority: 0
                 invalid_topic: 0
              invalid_priority: 0
                 duplicate_ids: 0
                empty_subjects: 0
                  empty_bodies: 45
   both_subject_and_body_empty: 0

Topic distribution:
topic
course-exam        267
deadline-action    197
event              189
administrative     159
advertisement      107

Priority distribution:
priority
Low       338
Medium    309
High      272


## Step 4 — Justified row removals

Rows are dropped only for: missing/invalid topic, missing/invalid priority, duplicate `id` (keep first), or both subject and body empty. Rows with only one of subject/body empty are kept. Each removal is logged.

In [ ]:
df = df_raw.copy()
removals = []

mask_bad_topic = ~df["topic"].isin(VALID_TOPICS)
n = int(mask_bad_topic.sum())
if n:
    removals.append(("missing_or_invalid_topic", n))
    df = df.loc[~mask_bad_topic].copy()

mask_bad_prio = ~df["priority"].isin(VALID_PRIORITIES)
n = int(mask_bad_prio.sum())
if n:
    removals.append(("missing_or_invalid_priority", n))
    df = df.loc[~mask_bad_prio].copy()

mask_dup = df["id"].duplicated(keep="first")
n = int(mask_dup.sum())
if n:
    removals.append(("duplicate_id", n))
    df = df.loc[~mask_dup].copy()

mask_empty_both = is_blank(df["subject"]) & is_blank(df["body_plain"])
n = int(mask_empty_both.sum())
if n:
    removals.append(("subject_and_body_both_empty", n))
    df = df.loc[~mask_empty_both].copy()

print(f"Rows before: {len(df_raw)}")
print(f"Rows after : {len(df)}")
print(f"Removed    : {len(df_raw) - len(df)}")
print()
if removals:
    print("Removal breakdown:")
    for reason, count in removals:
        print(f"  - {reason:<32}: {count}")
else:
    print("No rows removed.")

Rows before: 919
Rows after : 919
Removed    : 0

No rows removed.


## Step 5 — Light multilingual text preprocessing

Cleaning is intentionally light because the corpus is mixed English/Italian. We lowercase, strip simple HTML leftovers, replace emails/URLs with `EMAIL`/`URL` tokens, collapse whitespace, and trim. We do **not** remove punctuation or digits, and do not stem, lemmatize, or remove stopwords.

In [ ]:
URL_RE = re.compile(r"\b(?:https?://|ftp://|www\.)\S+", flags=re.IGNORECASE) # URL pattern
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", flags=re.IGNORECASE) # email pattern
HTML_TAG_RE = re.compile(r"<[^>]+>") # simple HTML tag pattern
HTML_ENTITY_RE = re.compile(r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+);", flags=re.IGNORECASE) # HTMl entities
WS_RE = re.compile(r"\s+") # any sequence of whitespace characters


def clean_text(value: object) -> str:
    """Light multilingual cleaning safe for English + Italian text."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value)

    text = HTML_TAG_RE.sub(" ", text)
    text = HTML_ENTITY_RE.sub(" ", text)

    # Order matters: emails before URLs (emails contain no scheme,
    # but doing emails first avoids accidental URL matches on addresses).
    text = EMAIL_RE.sub(" EMAIL ", text)
    text = URL_RE.sub(" URL ", text)

    text = text.lower()
    text = WS_RE.sub(" ", text).strip()
    return text

hi mario , please visit url or write to email . vedi anche url — grazie!!!


## Step 6 — Build `text_subject` and `text_body`

We apply `clean_text` to `subject` and `body_plain` and store the results in two new columns.

In [ ]:
df["text_subject"] = df["subject"].map(clean_text)
df["text_body"] = df["body_plain"].map(clean_text)

print("Lengths (chars):")
print(df[["text_subject", "text_body"]].apply(lambda c: c.str.len()).describe().round(1))

df[["id", "text_subject", "text_body"]].head(3)

Lengths (chars):
       text_subject  text_body
count         919.0      919.0
mean           62.1     1254.0
std            28.1     1148.8
min             6.0        0.0
25%            42.0      578.0
50%            59.0      916.0
75%            79.0     1595.0
max           171.0    13789.0


,id,text_subject,text_body
0,4e5d730277a0ddab,benvenuto nel servizio di posta dell'universita' di genova,"[questo e' un messaggio automatico, si prega di non rispondere] benvenuto nel servizio di posta dell'universita' di ..."
1,d8a3699e03d7335a,badge – student card [ts#142583742],"gentile studente, il tuo badge è disponibile. lo potrai ritirare presso il settore welcome office in piazza nunziata..."
2,2867eb3c4849e4fb,"aulaweb2024 90498: ml exam on thursday jan, 16th","90498 -> forum -> announcements -> ml exam on thursday jan, 16th url ml exam on thursday jan, 16th di nicoletta noce..."


## Step 7 — Build `text_subject_body`

Concatenate the two fields with `[SUBJECT] ... [BODY] ...` markers; an empty side is omitted.

In [ ]:
def join_subject_body(subject: str, body: str) -> str:
    parts = []
    if subject:
        parts.append(f"[SUBJECT] {subject}")
    if body:
        parts.append(f"[BODY] {body}")
    return " ".join(parts)


df["text_subject_body"] = [
    join_subject_body(s, b) for s, b in zip(df["text_subject"], df["text_body"])
]

print("Sample combined text:")
print(df["text_subject_body"].iloc[0][:300])
print("...")
print()
print("Empty combined texts:", int((df["text_subject_body"].str.len() == 0).sum()))

Sample combined text:


[SUBJECT] benvenuto nel servizio di posta dell'universita' di genova [BODY] [questo e' un messaggio automatico, si prega di non rispondere] benvenuto nel servizio di posta dell'universita' di genova. la casella di posta elettronica email e' attiva; puo' accedere con le sue credenziali unigepass. tut
...

Empty combined texts: 0


## Step 8 — Final post-cleaning audit

Re-runs the integrity checks on the cleaned dataframe as a closing report.

In [ ]:
final_report = {
    "rows": len(df),
    "missing_topic": int(df["topic"].isna().sum()),
    "missing_priority": int(df["priority"].isna().sum()),
    "invalid_topic": int((~df["topic"].isin(VALID_TOPICS)).sum()),
    "invalid_priority": int((~df["priority"].isin(VALID_PRIORITIES)).sum()),
    "duplicate_ids": int(df["id"].duplicated().sum()),
    "empty_text_subject": int((df["text_subject"].str.len() == 0).sum()),
    "empty_text_body": int((df["text_body"].str.len() == 0).sum()),
    "empty_text_subject_body": int((df["text_subject_body"].str.len() == 0).sum()),
}

print("Final integrity report")
print("-" * 40)
for k, v in final_report.items():
    print(f"{k:>26}: {v}")

print("\nFinal topic distribution:")
print(df["topic"].value_counts().to_string())
print("\nFinal priority distribution:")
print(df["priority"].value_counts().to_string())

Final integrity report
----------------------------------------
                      rows: 919
             missing_topic: 0
          missing_priority: 0
             invalid_topic: 0
          invalid_priority: 0
             duplicate_ids: 0
        empty_text_subject: 0
           empty_text_body: 45
   empty_text_subject_body: 0

Final topic distribution:
topic
course-exam        267
deadline-action    197
event              189
administrative     159
advertisement      107

Final priority distribution:
priority
Low       338
Medium    309
High      272


## Step 9 — Freeze and save

Persist the frozen dataset to `frozen_dataset.csv` (original keys/labels plus the three text variants). This is the artifact all downstream notebooks load.

In [ ]:
final_columns = [
    "id",
    "subject",
    "body_plain",
    "topic",
    "priority",
    "text_subject",
    "text_body",
    "text_subject_body",
]

df_final = df[final_columns].reset_index(drop=True)
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(df_final)} rows to {OUTPUT_PATH.resolve()}")
df_final.head(3)

Saved 919 rows to D:\UniGe\3\NLP\NLP-project\Preprocessing\frozen_dataset.csv

,id,subject,body_plain,topic,priority,text_subject,text_body,text_subject_body
0,4e5d730277a0ddab,Benvenuto nel servizio di posta dell'Universita' di Genova,"[Questo e' un messaggio automatico, si prega di non rispondere]\r\n \r\nBenvenuto nel servizio di pos...",administrative,Medium,benvenuto nel servizio di posta dell'universita' di genova,"[questo e' un messaggio automatico, si prega di non rispondere] benvenuto nel servizio di posta dell'universita' di ...","[SUBJECT] benvenuto nel servizio di posta dell'universita' di genova [BODY] [questo e' un messaggio automatico, si p..."
1,d8a3699e03d7335a,Badge – Student card [TS#142583742],"Gentile studente,\r\n\r\nil tuo badge è disponibile. Lo potrai ritirare presso il Settore Welcome\r\nOffice in Piazz...",administrative,Medium,badge – student card [ts#142583742],"gentile studente, il tuo badge è disponibile. lo potrai ritirare presso il settore welcome office in piazza nunziata...","[SUBJECT] badge – student card [ts#142583742] [BODY] gentile studente, il tuo badge è disponibile. lo potrai ritirar..."
2,2867eb3c4849e4fb,"AulaWeb2024 90498: ML exam on Thursday Jan, 16th","90498 -> Forum -> Announcements -> ML exam on Thursday Jan, 16th\r\nhttps://2024.aulaweb.unige.it/mod/forum/discuss....",course-exam,High,"aulaweb2024 90498: ml exam on thursday jan, 16th","90498 -> forum -> announcements -> ml exam on thursday jan, 16th url ml exam on thursday jan, 16th di nicoletta noce...","[SUBJECT] aulaweb2024 90498: ml exam on thursday jan, 16th [BODY] 90498 -> forum -> announcements -> ml exam on thur..."
